# ST456 Colab 一键运行 Notebook（VS Code 本地连接版）

适用于 VS Code 通过 SSH 连接 Colab 运行时的场景：

- 项目文件已在 Colab VM 上（通过 VS Code 上传）
- 自动检测项目目录，也可手动指定
- 结果直接在 VS Code 文件浏览器中查看
- E1-E5 是主实验，retrieval 只作为 appendix

In [ ]:
# ===== 参数区：运行前先改这里 =====

# 项目目录：留空则自动检测，或手动指定完整路径
PROJECT_ROOT = ''  # 例如 '/content/ST456group/codex-novel-continuation'

QUICK_VALIDATION = True
RUN_FULL_MAINLINE = True
RUN_APPENDIX_RETRIEVAL = False

print('参数已加载。')

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['PYTHONUNBUFFERED'] = '1'

def run(cmd, cwd=None):
    print(f'\n>>> {cmd}', flush=True)
    process = subprocess.Popen(
        cmd, shell=True, cwd=cwd,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in iter(process.stdout.readline, ''):
        print(line, end='', flush=True)
    process.wait()
    if process.returncode != 0:
        raise RuntimeError(f'命令执行失败（exit code {process.returncode}）：{cmd}')

def find_project_root():
    """自动检测 codex-novel-continuation 目录。"""
    search_roots = [Path.cwd(), Path('/content'), Path.home()]
    for root in search_roots:
        for candidate in root.rglob('codex-novel-continuation'):
            if candidate.is_dir() and (candidate / 'src').exists():
                return candidate
    return None

if PROJECT_ROOT:
    project_root = Path(PROJECT_ROOT)
else:
    print('自动检测项目目录...')
    project_root = find_project_root()
    if project_root is None:
        raise FileNotFoundError(
            '未找到 codex-novel-continuation 目录。\n'
            '请在参数区手动设置 PROJECT_ROOT。'
        )

if not project_root.exists():
    raise FileNotFoundError(f'项目目录不存在: {project_root}')

os.chdir(project_root)
print('项目目录:', Path.cwd())
run(f'{sys.executable} -m pip install -r requirements.txt')
print('环境准备完成。')

## 数据准备与 token 预算检查

In [ ]:
run(f'{sys.executable} scripts/download_gutenberg.py')
run(f'{sys.executable} scripts/build_dataset.py --context-size 4 --min-chars 40')

Path('artifacts/eval').mkdir(parents=True, exist_ok=True)
token_stat_configs = [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml'),
    ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml'),
    ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml'),
]
for experiment_id, config_path in token_stat_configs:
    output_path = Path('artifacts/eval') / f'token_stats_{experiment_id.lower()}.json'
    run(f'{sys.executable} scripts/inspect_token_stats.py --config {config_path} --output-path {output_path}')

print('Token 统计文件:')
for path in sorted(Path('artifacts/eval').glob('token_stats_*.json')):
    print('-', path)

## 主实验训练

E1 一定会跑。如果 `RUN_FULL_MAINLINE = True`，会继续跑 E2-E5。

In [ ]:
import gc, torch
sys.path.insert(0, str(Path.cwd() / 'src'))

from novel_continuation.training import load_training_config, train_baseline_model, train_retrieval_model

main_experiments = [
    ('E1', 'configs/e1_distilgpt2_plain_full.yaml', 'artifacts/e1_plain_full'),
]
if RUN_FULL_MAINLINE:
    main_experiments.extend([
        ('E2', 'configs/e2_distilgpt2_structured_full.yaml', 'artifacts/e2_structured_full'),
        ('E3', 'configs/e3_distilgpt2_structured_long_context.yaml', 'artifacts/e3_long_context'),
        ('E4', 'configs/e4_distilgpt2_structured_lora.yaml', 'artifacts/e4_lora'),
        ('E5', 'configs/e5_distilgpt2_structured_aux_ranking.yaml', 'artifacts/e5_aux_ranking'),
    ])

for experiment_id, config_path, _model_dir in main_experiments:
    print(f'\n===== 开始训练 {experiment_id} =====')
    config = load_training_config(Path(config_path))
    if QUICK_VALIDATION:
        config['num_train_epochs'] = 1
        print(f'[快速验证模式] {experiment_id}: epochs=1')
    if config.get('use_retrieval', False):
        trainer, tokenizer = train_retrieval_model(config)
    else:
        trainer, tokenizer = train_baseline_model(config)
    del trainer, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free_mb = torch.cuda.mem_get_info()[0] / 1024 / 1024
        print(f'{experiment_id} 训练完成，GPU 显存已释放（剩余 {free_mb:.0f} MB）')
    else:
        print(f'{experiment_id} 训练完成。')

print('\n主实验训练完成。')

## 样本生成、自动评估与人评导出

这一步会为每个已经训练完成的主实验生成：

- `generated_samples_*.jsonl`
- `metrics_*.csv`
- `human_eval_*.csv`

In [ ]:
from novel_continuation.training import load_trained_model_and_tokenizer
from novel_continuation.evaluation import evaluate_generated_rows, write_metrics_csv

sys.path.insert(0, str(Path.cwd() / 'scripts'))
from generate_samples import generate_rows, write_jsonl, load_jsonl
from prepare_human_eval import build_human_eval_rows, write_human_eval_csv

test_rows = load_jsonl(Path('data/processed/test.jsonl'))

if QUICK_VALIDATION:
    test_rows = test_rows[:20]
    print(f'[快速验证模式] 仅使用前 {len(test_rows)} 条测试样本')

for experiment_id, _config_path, model_dir in main_experiments:
    sample_path = Path('artifacts/eval') / f'generated_samples_{experiment_id.lower()}.jsonl'
    metrics_path = Path('artifacts/eval') / f'metrics_{experiment_id.lower()}.csv'
    human_eval_path = Path('artifacts/eval') / f'human_eval_{experiment_id.lower()}.csv'

    print(f'\n----- {experiment_id}: 生成样本 -----')
    generated = generate_rows(
        rows=test_rows,
        model_dir=Path(model_dir),
        max_new_tokens=80,
        use_retrieval=False,
    )
    write_jsonl(generated, sample_path)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f'----- {experiment_id}: 自动评估 -----')
    model, tokenizer = load_trained_model_and_tokenizer(Path(model_dir))
    metrics = evaluate_generated_rows(generated, model=model, tokenizer=tokenizer)
    write_metrics_csv(metrics, metrics_path)
    print(f'{experiment_id} 指标: ' + ', '.join(f'{k}={v:.4f}' for k, v in metrics.items()))

    del model, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f'----- {experiment_id}: 人评导出 -----')
    human_rows = build_human_eval_rows(generated, system_label=experiment_id)
    write_human_eval_csv(human_rows, human_eval_path)
    print(f'{experiment_id} 评估全部完成。')

print('\n评估文件已生成。')
for path in sorted(Path('artifacts/eval').glob('*')):
    print('-', path)

## Appendix：retrieval 附加实验

只有在 `RUN_APPENDIX_RETRIEVAL = True` 时才会执行这一块。

In [ ]:
if RUN_APPENDIX_RETRIEVAL:
    run(f'{sys.executable} scripts/train_retrieval_model.py --config configs/retrieval_distilgpt2.yaml')
    run(f'{sys.executable} scripts/generate_samples.py --model-dir artifacts/retrieval --use-retrieval --history-path data/processed/train.jsonl --output-path artifacts/eval/generated_samples_retrieval.jsonl')
    run(f'{sys.executable} scripts/run_auto_eval.py --model-dir artifacts/retrieval --generated-path artifacts/eval/generated_samples_retrieval.jsonl --output-path artifacts/eval/metrics_retrieval.csv')
    run(f'{sys.executable} scripts/prepare_human_eval.py --input-path artifacts/eval/generated_samples_retrieval.jsonl --output-path artifacts/eval/human_eval_retrieval.csv --system-label Retrieval')
    print('retrieval 附加实验已完成。')
else:
    print('已跳过 retrieval 附加实验。')

## 清理 checkpoint 并汇总结果

清理中间 checkpoint 节省磁盘空间，结果文件在 `artifacts/eval/` 目录下，可直接通过 VS Code 文件浏览器查看。

In [ ]:
for ckpt_dir in sorted(Path('artifacts').rglob('checkpoint-*')):
    if ckpt_dir.is_dir():
        shutil.rmtree(ckpt_dir)
        print(f'已清理: {ckpt_dir}')

print('\n===== 全部完成 =====')
print(f'结果目录: {Path.cwd() / "artifacts" / "eval"}')
print('\n生成的文件:')
for path in sorted(Path('artifacts/eval').glob('*')):
    size_kb = path.stat().st_size / 1024
    print(f'  {path.name}  ({size_kb:.1f} KB)')
print('\n可直接在 VS Code 文件浏览器中查看 artifacts/eval/ 目录。')